# 02_v2 — Title-Guided Evidence Attention Two-Tower Classifier
### Bengali, Banglish & English Clickbait Detection with Fine-Grained Claim Verification
**Hardware Target:** Kaggle Dual GPU (T4 x 2) or Google Colab (Free T4)

---

### Why Title-Guided Evidence Attention?
Standard Siamese / Two-Tower models encode the entire article body into a single `[CLS]` token ($h_c$). However, a 512-token news body covers background history, quotes, and context, causing $h_c$ to become a **blurry global average**. When a sensational headline misrepresents just one detail, comparing the headline to a blurry global average dilutes the mismatch signal.

In this **v2 Architecture**:
1. **Title-Guided Evidence Projection**: Multi-Head Cross-Attention ($Q = T_{\text{emb}}, K = C_{\text{emb}}, V = C_{\text{emb}}$) projects body evidence into the **exact sequence dimension of the title** ($128 \times 768$). Each title token position receives the specific body evidence retrieved for that exact claim token.
2. **Fine-Grained Verification**: Computes claim-vs-retrieved evidence cosine $\cos(h_{\text{title}}, h_{\text{evidence}})$, discrepancy $|h_t - h_e|$, and interaction $h_t \odot h_e$.
3. **Token-Level Unsupported Word Detection**: Computes $\min_i \cos(T_i, A_{\text{ev}, i})$ across title tokens to catch sensational clickbait "bait words" that lack any textual backing in the article.
4. **Macro Full Content Connection**: The global document representation $h_{\text{content\_global}}$ connects directly into the final classification FFN so macro article context is never lost.

```
Title (128 toks) ──────▶ [Shared BanglishBERT] ──▶ T_emb (B, 128, 768) ──┐
                                                                         │
Body (512 toks)  ──────▶ [Shared BanglishBERT] ──▶ C_emb (B, 512, 768) ──┤
                                                   │                     │
                                                   │    Cross-Attention: │
                                                   │    Q = T_emb (128)  │
                                                   └──▶ K = C_emb (512) ─┴─▶ A_ev (B, 128, 768)
                                                        V = C_emb (512)      (Title-dimension aligned)
                                                                             │
      ┌──────────────────────────────────────────────────────────────────────┴──────────────┐
      │ Feature Extraction & Alignment Verification                                         │
      │ 1. h_title = MeanPool(T_emb)                                   (768-d)              │
      │ 2. h_evidence = MeanPool(A_ev)                                 (768-d)              │
      │ 3. h_content_global = MeanPool(C_emb)                          (768-d) [Full Body]  │
      │ 4. Evidence Discrepancy = |h_title - h_evidence|               (768-d)              │
      │ 5. Evidence Interaction = h_title * h_evidence                 (768-d)              │
      │ 6. cos_evidence = cos(h_title, h_evidence)                     (1-d)                │
      │ 7. cos_global = cos(h_title, h_content_global)                 (1-d)                │
      │ 8. min_token_cos = min_i cos(T_i, A_ev_i)                      (1-d) [Bait Detector]│
      └──────────────────────────────────────────────────────────────────────┬──────────────┘
                                                                             ▼
                                         Concatenation: 5 × 768 + 3 = 3,843 dimensions
                                                                             │
                                              Linear(3843 ─▶ 768) + GELU + Dropout(0.1)
                                                                             │
                                              Linear(768 ─▶ 384) + GELU + Dropout(0.1)
                                                                             │
                                              Linear(384 ─▶ 1) ──▶ isClickBait Logit
```


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q transformers datasets scikit-learn pandas matplotlib


## 1b. (Optional) Mount Drive for persistent data + checkpoints
Colab wipes `/content` on recycle. Mounting Drive lets `splits/` and `outputs/` survive disconnects.
Skip this cell if you will upload `splits/` directly in the next step instead.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')  # authorize once; files live under /content/drive/MyDrive/
    print('Drive mounted. Put splits/ at: /content/drive/MyDrive/declickbait/splits/')
except ImportError:
    print('Not running in Colab; skipping Drive mount.')


In [ ]:
import glob, json, os, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| gpus:', torch.cuda.device_count())

def find_data_root():
    cands = [
        os.environ.get('DECLICKBAIT_DATA', ''),
        '/content/drive/MyDrive/declickbait/splits', '/content/drive/MyDrive/splits',
        '/kaggle/input/declickbait-splits/splits', '/kaggle/input/declickbait-splits',
        '/kaggle/input/declickbait/splits', '/kaggle/input/declickbait',
        'splits', '../splits', '/kaggle/working/splits', '/content/splits'
    ]
    cands += glob.glob('/kaggle/input/*/splits') + glob.glob('/kaggle/input/*')
    for c in cands:
        if not c or not Path(c).exists():
            continue
        if (Path(c) / 'train_train_val_test.jsonl').exists():
            return str(c)
        if (Path(c) / 'splits' / 'train_train_val_test.jsonl').exists():
            return str(Path(c) / 'splits')
    return None

DATA_ROOT = find_data_root()
print('DATA_ROOT:', DATA_ROOT)
assert DATA_ROOT, 'splits/ not found. Upload splits/ or attach declickbait-splits dataset.'

OUT = '/kaggle/working/outputs' if Path('/kaggle/working').exists() else ('/content/outputs' if Path('/content').exists() else 'outputs')
Path(OUT).mkdir(parents=True, exist_ok=True)
print('OUT:', OUT)

def print_mem(tag):
    if torch.cuda.is_available():
        n = max(torch.cuda.device_count(), 1)
        print(f"[VRAM {tag}] used={torch.cuda.memory_allocated() / 1e9:.2f}GB reserved={torch.cuda.memory_reserved() / 1e9:.2f}GB / {16 * n}GB")


In [ ]:
import re as _re
PROMO_PATTERNS = [
    r'subscrib\w*', r'bell\s*icon', r'notification\s*(bell|on)',
    r'\blike\b.{0,20}\bcomment\b.{0,20}\bshare\b', r'\bshare\b.{0,20}\bsubscrib',
    r'watch\s+more', r'click\s+here', r'download\s+(our\s+)?app',
    r'facebook\.com', r'youtube\.com', r'bit\.ly', r'whatsapp', r't\.me/',
    r'hotline', r'sponsor', r'promo', r'official\s+(youtube\s+)?channel',
    r'for\s+more\s+videos?', r'stay\s+tuned',
    r'সাবস্ক্রাইব', r'সাবসক্রাইব', r'চ্যানেলটি', r'বেল\s*(আইকন|বাটন)', r'নোটিফিকেশন',
    r'লাইক\s*(করুন|দিন)', r'শেয়ার\s*করুন', r'কমেন্ট\s*করুন', r'আমাদের\s+চ্যানেল',
]
_PROMO_RX = _re.compile('|'.join(f'(?:{p})' for p in PROMO_PATTERNS), _re.IGNORECASE)
_SENT_RX = _re.compile(r'(?<=[.!?।…])\s+|\n+')

def clean_body(text, max_chars=3500, min_keep=50):
    # Runtime promo strip; dataset files untouched
    if not text:
        return ''
    t = str(text).strip()
    kept = [s for s in _SENT_RX.split(t) if s.strip() and not _PROMO_RX.search(s)]
    out = ' '.join(kept).strip()
    return out[:max_chars].strip() if len(out) >= min_keep else t[:max_chars].strip()

def body_of(r, n=3500):
    raw = str(r.get('body') or r.get('content_text') or r.get('transcript') or '')
    return clean_body(raw, max_chars=n)
print('helpers ready')


In [ ]:
# ---- TUNABLE CONFIG (Kaggle T4 x 2: 32GB total VRAM | Colab T4: 15GB) ----
MODEL_NAME = 'csebuetnlp/banglishbert'
N_GPUS = max(torch.cuda.device_count(), 1)
BATCH_SIZE = 24 * N_GPUS   # 48 on Kaggle Dual T4 (24/GPU), 24 on Colab T4
EVAL_BATCH = 48 * N_GPUS   # 96 on Kaggle Dual T4, 48 on Colab T4
TT_TLEN = 128              # Title token sequence budget
TT_BLEN = 512              # Body content token sequence budget (640 total tokens)
EPOCHS = 4
LR = 2e-5
WORKERS = 2
print(f'Config: N_GPUS={N_GPUS} | BATCH={BATCH_SIZE} | EVAL_BATCH={EVAL_BATCH} | TT_TLEN={TT_TLEN} | TT_BLEN={TT_BLEN}')


In [ ]:
from transformers import AutoTokenizer

def load_jsonl(path):
    with open(path, encoding='utf-8', errors='replace') as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl(f'{DATA_ROOT}/train_train_val_test.jsonl')
val_rows = load_jsonl(f'{DATA_ROOT}/val_train_val_test.jsonl')
test_rows = load_jsonl(f'{DATA_ROOT}/test_train_val_test.jsonl')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class TwoTowerV2Dataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        titles = [str(r.get('original_title') or '').strip() for r in rows]
        bodies = [body_of(r, 3500) for r in rows]
        self.te = tok(titles, truncation=True, padding=True, max_length=TT_TLEN)
        self.ce = tok(bodies, truncation=True, padding=True, max_length=TT_BLEN)
        self.labels = [
            1.0 if str(r.get('clickbait', False)).lower() in ('1', 'true', 't', 'yes', 'y') else 0.0
            for r in rows
        ]
        self.langs = [str(r.get('language_mix', '?')) for r in rows]
        self.raw_titles = titles
        self.raw_bodies = bodies

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            't_ids': torch.tensor(self.te['input_ids'][i], dtype=torch.long),
            't_mask': torch.tensor(self.te['attention_mask'][i], dtype=torch.long),
            'c_ids': torch.tensor(self.ce['input_ids'][i], dtype=torch.long),
            'c_mask': torch.tensor(self.ce['attention_mask'][i], dtype=torch.long),
            'y': torch.tensor(self.labels[i], dtype=torch.float32)
        }

tt_train = TwoTowerV2Dataset(train_rows)
tt_val = TwoTowerV2Dataset(val_rows)
tt_test = TwoTowerV2Dataset(test_rows)
print(f'Pairs train/val/test: {len(tt_train)}/{len(tt_val)}/{len(tt_test)}')


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class TitleGuidedTwoTowerHead(nn.Module):
    """
    1. Projects Content into Title dimension via Multi-Head Cross-Attention (Q=Title, K=V=Content).
    2. Computes fine-grained Claim-vs-Evidence alignment (evidence cosine, diff, interaction).
    3. Detects unsupported sensational words via token-level min-cosine.
    4. Connects macro full-content context into the final classification FFN.
    """
    def __init__(self, hidden=768, heads=8):
        super().__init__()
        # Title queries Content -> Output has shape (B, TT_TLEN, hidden)
        self.cross_attn = nn.MultiheadAttention(hidden, heads, batch_first=True)
        
        # Feature space:
        # - h_title (H)
        # - h_evidence (H) [body content retrieved specifically for title tokens]
        # - h_content_global (H) [full body global context]
        # - diff_evidence: |h_title - h_evidence| (H)
        # - prod_evidence: h_title * h_evidence (H)
        # - cos_evidence: cos(h_title, h_evidence) (1)
        # - cos_global: cos(h_title, h_content_global) (1)
        # - min_tok_cos: min_i cos(T_i, A_ev_i) (1) [unsupported word detector]
        # Total input dim = 5 * H + 3 = 5 * 768 + 3 = 3,843
        in_dim = hidden * 5 + 3
        self.fuse = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, t_emb, t_mask, c_emb, c_mask, return_diagnostics: bool = False):
        # 1. Content is attended into the exact sequence length of Title: (B, 128, H)
        a_ev, attn_weights = self.cross_attn(
            query=t_emb,
            key=c_emb,
            value=c_emb,
            key_padding_mask=~c_mask.bool(),
            need_weights=return_diagnostics
        )
        
        # 2. Masked mean pooling
        t_weights = t_mask.unsqueeze(-1).float()
        c_weights = c_mask.unsqueeze(-1).float()
        
        h_title = (t_emb * t_weights).sum(1) / t_weights.sum(1).clamp_min(1)
        h_evidence = (a_ev * t_weights).sum(1) / t_weights.sum(1).clamp_min(1)
        h_content_global = (c_emb * c_weights).sum(1) / c_weights.sum(1).clamp_min(1)
        
        # 3. Fine-grained claim vs. evidence features
        diff_evidence = (h_title - h_evidence).abs()
        prod_evidence = h_title * h_evidence
        cos_evidence = F.cosine_similarity(h_title, h_evidence, dim=-1).unsqueeze(-1)
        
        # 4. Macro full content cosine
        cos_global = F.cosine_similarity(h_title, h_content_global, dim=-1).unsqueeze(-1)
        
        # 5. Token-level unsupported word detector (min cosine across non-padding title tokens)
        tok_cos = F.cosine_similarity(t_emb, a_ev, dim=-1)  # (B, 128)
        tok_cos_masked = tok_cos.masked_fill(~t_mask.bool(), 1.0)
        min_tok_cos = tok_cos_masked.min(dim=-1).values.unsqueeze(-1)  # (B, 1)
        
        # 6. Concatenate fine evidence + macro full content features
        feat = torch.cat([
            h_title, 
            h_evidence, 
            h_content_global, 
            diff_evidence, 
            prod_evidence, 
            cos_evidence, 
            cos_global, 
            min_tok_cos
        ], dim=-1)
        
        logits = self.fuse(feat).squeeze(-1)
        if return_diagnostics:
            return logits, cos_evidence.squeeze(-1), cos_global.squeeze(-1), tok_cos_masked
        return logits


In [ ]:
enc = AutoModel.from_pretrained(MODEL_NAME).to(device)
H = enc.config.hidden_size
head = TitleGuidedTwoTowerHead(H).to(device)

n_params = sum(p.numel() for p in enc.parameters()) + sum(p.numel() for p in head.parameters())
print(f'Model loaded: {MODEL_NAME} + TitleGuidedTwoTowerHead ({n_params/1e6:.1f}M params)')
print_mem('model initialized')

if torch.cuda.device_count() > 1:
    print(f'⚡ Kaggle Dual T4 detected! Parallelizing Two-Tower v2 across {torch.cuda.device_count()} GPUs.')
    enc_dp = torch.nn.DataParallel(enc)
    head_dp = torch.nn.DataParallel(head)
else:
    enc_dp, head_dp = enc, head
raw_enc, raw_head = enc, head


In [ ]:
from tqdm.auto import tqdm
import time
from sklearn.metrics import accuracy_score, f1_score

opt = torch.optim.AdamW(list(raw_enc.parameters()) + list(raw_head.parameters()), lr=LR, weight_decay=0.01)
loss_fn = torch.nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
BEST_CKPT = f'{OUT}/cls_banglishbert_v2_best.pt'

# Pre-tokenized in RAM -> num_workers=0 eliminates IPC serialization overhead & prevents Colab thread stalls
def get_loader(ds, shuffle, batch_sz):
    return torch.utils.data.DataLoader(
        ds, batch_size=batch_sz, shuffle=shuffle, num_workers=0, pin_memory=True
    )

tr_ld = get_loader(tt_train, True, BATCH_SIZE)
va_ld = get_loader(tt_val, False, EVAL_BATCH)

best_f1, best_state = -1.0, None
n_batches = len(tr_ld)

print(f"🚀 Starting accelerated training on T4 (FP16 Tensor Cores active) | {n_batches} batches/epoch")

for epoch in range(EPOCHS):
    enc_dp.train(); head_dp.train()
    tot_loss = 0.0
    ep_start = time.time()
    
    # Visual Interactive % Progress Bar (Updates in-place in Colab)
    pbar = tqdm(tr_ld, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch", dynamic_ncols=True)
    
    for step, b in enumerate(pbar, start=1):
        t_ids, t_m = b['t_ids'].to(device), b['t_mask'].to(device)
        c_ids, c_m = b['c_ids'].to(device), b['c_mask'].to(device)
        y = b['y'].to(device)
        
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            t_emb = enc_dp(t_ids, attention_mask=t_m).last_hidden_state
            c_emb = enc_dp(c_ids, attention_mask=c_m).last_hidden_state
            logits = head_dp(t_emb, t_m, c_emb, c_m)
            loss = loss_fn(logits, y).mean()
        
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(list(raw_enc.parameters()) + list(raw_head.parameters()), 1.0)
        scaler.step(opt)
        scaler.update()
        
        tot_loss += loss.item()
        pbar.set_postfix({'loss': f'{tot_loss/step:.4f}'})

    # Validation loop with visual progress bar
    enc_dp.eval(); head_dp.eval()
    val_probs, val_true = [], []
    val_bar = tqdm(va_ld, desc=f"Validating Ep {epoch+1}", unit="batch", leave=False, dynamic_ncols=True)
    with torch.no_grad():
        for b in val_bar:
            t_ids, t_m = b['t_ids'].to(device), b['t_mask'].to(device)
            c_ids, c_m = b['c_ids'].to(device), b['c_mask'].to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                t_emb = enc_dp(t_ids, attention_mask=t_m).last_hidden_state
                c_emb = enc_dp(c_ids, attention_mask=c_m).last_hidden_state
                logits = head_dp(t_emb, t_m, c_emb, c_m)
            val_probs.extend(torch.sigmoid(logits).cpu().tolist())
            val_true.extend(b['y'].tolist())

    # Threshold sweep on validation set
    best_tau, best_val_f1 = 0.5, 0.0
    for tau in np.linspace(0.3, 0.7, 41):
        f1_cand = f1_score(val_true, [1 if p >= tau else 0 for p in val_probs], zero_division=0)
        if f1_cand > best_val_f1:
            best_val_f1, best_tau = f1_cand, tau

    avg_loss = tot_loss / max(n_batches, 1)
    ep_sec = time.time() - ep_start
    print(f'=== Epoch {epoch+1}/{EPOCHS} Done in {int(ep_sec//60)}m{int(ep_sec%60):02d}s | loss={avg_loss:.4f} | val_f1={best_val_f1:.4f} (at tau*={best_tau:.2f}) ===')

    if best_val_f1 > best_f1:
        best_f1 = best_val_f1
        best_state = {
            'enc': {k: v.cpu() for k, v in raw_enc.state_dict().items()},
            'head': {k: v.cpu() for k, v in raw_head.state_dict().items()},
            'best_tau': best_tau,
            'val_f1': best_f1
        }
        torch.save(best_state, BEST_CKPT)
        print(f'-> 💾 Saved new best checkpoint to {BEST_CKPT} (val_f1={best_f1:.4f})')

print_mem('training finished')
print(f'Saved best model (val_f1={best_f1:.4f}) to {BEST_CKPT}')


In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

best_ckpt = torch.load(BEST_CKPT, map_location=device)
raw_enc.load_state_dict({k: v.to(device) for k, v in best_ckpt['enc'].items()})
raw_head.load_state_dict({k: v.to(device) for k, v in best_ckpt['head'].items()})
opt_tau = best_ckpt.get('best_tau', 0.5)
print(f'Loaded best weights (optimal decision threshold tau*={opt_tau:.2f})')

te_ld = get_loader(tt_test, False, EVAL_BATCH)
raw_enc.eval(); raw_head.eval()

test_probs, test_true = [], []
test_bar = tqdm(te_ld, desc="Evaluating Test Set", unit="batch", dynamic_ncols=True)
with torch.no_grad():
    for b in test_bar:
        t_ids, t_m = b['t_ids'].to(device), b['t_mask'].to(device)
        c_ids, c_m = b['c_ids'].to(device), b['c_mask'].to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            t_emb = raw_enc(t_ids, attention_mask=t_m).last_hidden_state
            c_emb = raw_enc(c_ids, attention_mask=c_m).last_hidden_state
            logits = raw_head(t_emb, t_m, c_emb, c_m)
        test_probs.extend(torch.sigmoid(logits).cpu().tolist())
        test_true.extend(b['y'].tolist())

test_preds = [1 if p >= opt_tau else 0 for p in test_probs]
acc = accuracy_score(test_true, test_preds)
f1 = f1_score(test_true, test_preds, zero_division=0)
auc = roc_auc_score(test_true, test_probs)

print('=' * 50)
print(f'TEST RESULTS (Title-Guided Two-Tower v2):')
print(f'Accuracy: {acc:.4f} | F1-Score: {f1:.4f} | ROC-AUC: {auc:.4f}')
print('=' * 50)

# Per-language breakdown
langs = tt_test.langs
for lg in sorted(set(langs)):
    idx = [i for i, l in enumerate(langs) if l == lg]
    y_t_lg = [test_true[i] for i in idx]
    y_p_lg = [test_preds[i] for i in idx]
    flag = '  <-- zero-shot / eval-only' if lg == 'banglish' else ''
    print(f'{lg:10s} (n={len(idx):5d}): F1={f1_score(y_t_lg, y_p_lg, zero_division=0):.4f} | Acc={accuracy_score(y_t_lg, y_p_lg):.4f}{flag}')


In [ ]:
def diagnose_clickbait(title: str, body: str):
    """
    Analyzes a Title-Body pair and reveals:
    1. Predicted Clickbait Probability
    2. Fine-grained Claim-vs-Evidence Cosine (micro alignment)
    3. Global Content Cosine (macro document context)
    4. The exact token in the title with the least body support!
    """
    clean_b = clean_body(body, 3500)
    te = tok(title, truncation=True, padding=True, max_length=TT_TLEN, return_tensors='pt').to(device)
    ce = tok(clean_b, truncation=True, padding=True, max_length=TT_BLEN, return_tensors='pt').to(device)
    
    raw_enc.eval(); raw_head.eval()
    with torch.no_grad():
        t_emb = raw_enc(te['input_ids'], attention_mask=te['attention_mask']).last_hidden_state
        c_emb = raw_enc(ce['input_ids'], attention_mask=ce['attention_mask']).last_hidden_state
        logits, cos_ev, cos_glob, tok_cos = raw_head(
            t_emb, te['attention_mask'], c_emb, ce['attention_mask'], return_diagnostics=True
        )
        prob = float(torch.sigmoid(logits))
        cos_evidence = float(cos_ev[0])
        cos_global = float(cos_glob[0])
        token_scores = tok_cos[0].cpu().tolist()
    
    tokens = tok.convert_ids_to_tokens(te['input_ids'][0])
    valid_tokens = [
        (t, s) for t, s, m in zip(tokens, token_scores, te['attention_mask'][0])
        if m == 1 and t not in ('[CLS]', '[SEP]', '[PAD]')
    ]
    least_supported = min(valid_tokens, key=lambda x: x[1]) if valid_tokens else ('None', 1.0)
    
    return {
        'probability': prob,
        'is_clickbait': prob >= opt_tau,
        'claim_evidence_cosine': cos_evidence,
        'global_content_cosine': cos_global,
        'least_supported_word': least_supported[0],
        'word_support_score': least_supported[1]
    }
print('Diagnostic tool ready')


In [ ]:
import random

print('--- A) 3 Random Test Rows with Evidence Diagnostics ---')
for i in random.sample(range(len(tt_test)), 3):
    r_title = tt_test.raw_titles[i]
    r_body = tt_test.raw_bodies[i]
    label = int(tt_test.labels[i])
    res = diagnose_clickbait(r_title, r_body)
    print(f'TITLE: {r_title[:90]}')
    print(f"PRED:  p={res['probability']:.3f} (is_cb={res['is_clickbait']}) | TRUE: {label}")
    print(f"EVIDENCE COS: {res['claim_evidence_cosine']:.3f} | GLOBAL COS: {res['global_content_cosine']:.3f}")
    print(f"LEAST SUPPORTED TOKEN: '{res['least_supported_word']}' (support={res['word_support_score']:.3f})")
    print('-' * 60)

print('\n--- B) Interactive User Test (Edit & Run) ---')
OWN_TITLE = 'Shocking Miracle Cures All Diseases Overnight!'
OWN_BODY = 'A routine medical study was published discussing common seasonal influenza management and hydration.'
res_own = diagnose_clickbait(OWN_TITLE, OWN_BODY)
print(f'TITLE: {OWN_TITLE}')
print(f"RESULT: Clickbait={res_own['is_clickbait']} (p={res_own['probability']:.3f})")
print(f"Claim-Evidence Alignment: {res_own['claim_evidence_cosine']:.3f}")
print(f"Most Suspicious Word: '{res_own['least_supported_word']}' (support score={res_own['word_support_score']:.3f})")
